# Đo lường Hiệu năng Thực thi (Google Colab CPU-only)

Báo cáo này tổng hợp thời gian chạy toàn hệ thống, mức tiêu thụ RAM cực đại (`Peak RAM` qua `psutil`) và dung lượng lưu trữ đĩa của 3 bảng dữ liệu đầu ra trên môi trường **Google Colab CPU-only**.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
summary_path = ROOT / 'summaries/parsing_summary.json'

if not summary_path.exists():
    raise FileNotFoundError(f"Không tìm thấy tệp tổng kết tại: {summary_path}")

data = json.loads(summary_path.read_text(encoding='utf-8'))
env = data['environment']
perf = data['performance_metrics']
cov = data['coverage_statistics']

display(Markdown(f"### Môi trường đánh giá: `{env['platform']}`"))

## 1. Thông số Phần cứng & Môi trường Thực thi

Cấu hình máy ảo Google Colab CPU-only được hệ thống tự động nhận diện và lưu vết.

In [ ]:
env_df = pd.DataFrame([
    {'Thông số': 'Nền tảng thực thi', 'Giá trị': env['platform']},
    {'Thông số': 'Mô hình CPU', 'Giá trị': env['cpu_model']},
    {'Thông số': 'Số nhân CPU', 'Giá trị': f"{env['cpu_cores']} cores"},
    {'Thông số': 'Tổng RAM hệ thống', 'Giá trị': f"{env['total_ram_gb']} GB"},
    {'Thông số': 'Phiên bản Python', 'Giá trị': env['python_version']},
])
display(env_df)

## 2. Chỉ số Hiệu năng Cốt lõi

Tổng hợp thời gian chạy end-to-end, thời gian trung bình xử lý một bài báo, Peak RAM và dung lượng lưu trữ 3 bảng dữ liệu.

In [ ]:
perf_df = pd.DataFrame([
    {'Chỉ số hiệu năng': 'Thời gian chạy End-to-End', 'Giá trị': f"{perf['end_to_end_wall_time_sec']} giây ({round(perf['end_to_end_wall_time_sec']/60, 2)} phút)"},
    {'Chỉ số hiệu năng': 'Thời gian trung bình / bài báo', 'Giá trị': f"{perf['avg_time_per_publication_sec']} giây / paper"},
    {'Chỉ số hiệu năng': 'RAM tiêu thụ Cực đại (Peak RAM)', 'Giá trị': f"{perf['peak_ram_mb']} MB ({round(perf['peak_ram_mb']/1024, 2)} GB)"},
    {'Chỉ số hiệu năng': 'RAM tiêu thụ Trung bình (Avg RAM)', 'Giá trị': f"{perf['avg_ram_mb']} MB"},
    {'Chỉ số hiệu năng': 'Dung lượng lưu trữ đầu ra (3 bảng)', 'Giá trị': f"{perf['final_output_storage_mb']} MB"},
])
display(perf_df)

# Trực quan hóa các chỉ số tài nguyên và bộ nhớ
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# Biểu đồ RAM Footprint
ram_metrics = ['Average RAM', 'Peak RAM', 'Max System RAM']
ram_values = [perf['avg_ram_mb'], perf['peak_ram_mb'], env['total_ram_gb'] * 1024]
bars1 = ax1.bar(ram_metrics, ram_values, color=['#17becf', '#ff7f0e', '#7f7f7f'], width=0.55)
ax1.set_ylabel('Dung lượng RAM (MB)')
ax1.set_title('Mức Tiêu thụ RAM so với Giới hạn Colab', fontweight='bold')
ax1.grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars1:
    y = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, y + 100, f'{y:,.1f} MB', ha='center', va='bottom', fontweight='bold', fontsize=9)

# Biểu đồ Phân bổ Thời gian & Dung lượng
resources = ['End-to-End Time (s)', 'Storage Footprint (MB)']
values = [perf['end_to_end_wall_time_sec'], perf['final_output_storage_mb']]
bars2 = ax2.bar(resources, values, color=['#2ca02c', '#9467bd'], width=0.45)
ax2.set_title('Thời gian Thực thi & Dung lượng 3 Bảng', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars2:
    y = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, y + (max(values)*0.02), f'{y:,.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()